# Phase 3: GBIF Biodiversity Occurrence Data Collection & Validation
## Notebook 01: GBIF Source Investigation, API Architecture and Data Governance

### Research Context & Scope
The **Dark Ecology Intelligence (DEI)** project studies the ecological interactions between Artificial Light at Night (ALAN) and biodiversity across three distinct landscapes in Southern India:
1. **Bengaluru** — Urban core and expanding peri-urban fringe
2. **Mysuru** — Peri-urban and secondary urban landscape
3. **Nilgiri–Bandipur** — Tropical forest, national parks, and Western Ghats biodiversity hotspot

#### Governing Data Integrity Rules
- **ZERO SYNTHETIC DATA**: Strictly empirical biodiversity occurrence records collected from the Global Biodiversity Information Facility (GBIF). No fabricated coordinates, synthetic sightings, or invented absence points.
- **FOUR-DATASET FREEZE**: This phase works **exclusively with GBIF**. Other datasets (GloBI interactions, ESA WorldCover) remain strictly frozen and untouched.
- **AUDIT-BEFORE-DOWNLOAD**: Comprehensive architectural evaluation, parameter validation, and bias assessment before initiating any bulk download.

In [1]:
import os
import sys
import json
import requests
import pandas as pd
from datetime import datetime

print(f"Execution Timestamp: {datetime.now().isoformat()}")
print(f"Python Interpreter: {sys.executable}")

Execution Timestamp: 2026-11-03T21:04:28.159849
Python Interpreter: /Users/apple/Documents/5th sem el/ANN_PBL/dark_eco/.venv/bin/python3


### 1. Review of Existing Phase 1 Work & Earlier Probes
During Phase 1, preliminary exploratory probes were performed against the GBIF Occurrence API. We now inspect those earlier records and inventory the existing artifacts.

In [2]:
# Inspect Phase 1 checks summary
phase1_summary_path = os.path.join("..", "..", "data", "metadata", "phase_1_checks", "gbif_regional_summary.csv")
if os.path.exists(phase1_summary_path):
    df_p1 = pd.read_csv(phase1_summary_path)
    print("--- Phase 1 Preliminary GBIF Probe Summary ---")
    print(df_p1.to_string(index=False))
else:
    print("Phase 1 summary file not found at expected path.")

--- Phase 1 Preliminary GBIF Probe Summary ---
       region_id                                         region_name  total_gbif_records  records_1992_2013  records_2014_2024                                                               basis_of_record
       bengaluru                   Bengaluru urban-peri-urban window             3562914              25423            2941456 HUMAN_OBSERVATION: 3,558,000, PRESERVED_SPECIMEN: 3,689, MATERIAL_SAMPLE: 848
          mysuru             Mysuru peri-urban / transitional window              937355              72234             749703     HUMAN_OBSERVATION: 936,474, PRESERVED_SPECIMEN: 548, MATERIAL_SAMPLE: 177
nilgiri_bandipur Bandipur-Mudumalai forest window (Nilgiri-Bandipur)              240918              15448             187147   HUMAN_OBSERVATION: 238,208, PRESERVED_SPECIMEN: 2,410, MATERIAL_SAMPLE: 213


### 2. Official GBIF Architecture & Core API Endpoints
The Global Biodiversity Information Facility (GBIF) provides RESTful API services rooted at `https://api.gbif.org/v1/`.

| Endpoint Category | Base URL / Path | Primary Use Case in DEI Research | Rate Limits / Parameters |
| :--- | :--- | :--- | :--- |
| **Occurrence Search** | `/v1/occurrence/search` | Exploratory querying, spatial probing, facet counts | `limit <= 300`, `offset <= 100,000` |
| **Occurrence Count** | `/v1/occurrence/count` | Instant global or regional total record counts | High-speed cache lookup |
| **Download Request** | `/v1/occurrence/download/request` | Asynchronous bulk extraction generating citable DOI | Requires user authentication |
| **Species Backbone** | `/v1/species` | Taxonomic resolution against GBIF backbone | Real-time taxonomic key matching |

In [3]:
# Live GBIF API Health & Connectivity Check
headers = {"User-Agent": "DarkEcoResearch/1.0 (Academic ANN-PBL research)"}
resp = requests.get("https://api.gbif.org/v1/occurrence/search?limit=0", headers=headers, timeout=10)
print(f"GBIF API HTTP Status: {resp.status_code}")
data = resp.json()
print(f"Global Occurrence Volume indexed in GBIF: {data.get('count', 0):,} records")

GBIF API HTTP Status: 200
Global Occurrence Volume indexed in GBIF: 3,889,961,054 records


### 3. Darwin Core Standard Schema & Available Attributes
GBIF formats all occurrence records using the **Darwin Core Standard (DwC)**. The key attributes relevant to spatial ecology and nocturnal light modeling include:
- **Spatial Positioning**: `decimalLatitude`, `decimalLongitude`, `coordinateUncertaintyInMeters`, `geodeticDatum`
- **Temporal Stamping**: `eventDate`, `year`, `month`, `day`
- **Taxonomic Resolution**: `scientificName`, `taxonRank`, `kingdom`, `phylum`, `class`, `order`, `family`, `genus`, `species`
- **Observational Protocol**: `basisOfRecord`, `occurrenceStatus`, `individualCount`, `samplingProtocol`, `eventID`
- **Provenance & Integrity**: `datasetKey`, `institutionCode`, `collectionCode`, `recordedBy`, `issues`

In [4]:
# Inspect sample Darwin Core record from live API
sample_resp = requests.get("https://api.gbif.org/v1/occurrence/search?limit=1", headers=headers, timeout=10).json()
rec = sample_resp["results"][0]

sample_fields = ["key", "datasetKey", "basisOfRecord", "scientificName", "taxonRank", 
                 "kingdom", "phylum", "class", "order", "family", "genus", "species",
                 "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters",
                 "eventDate", "year", "month", "day", "occurrenceStatus", "individualCount", "issues"]

sample_dict = {f: rec.get(f) for f in sample_fields}
df_sample = pd.DataFrame(list(sample_dict.items()), columns=["Darwin Core Term", "Sample Value"])
print(df_sample.to_string(index=False))

             Darwin Core Term                                                                                                        Sample Value
                          key                                                                                                           182368733
                   datasetKey                                                                                857aa892-f762-11e1-a439-00145eb45e9a
                basisOfRecord                                                                                                         OBSERVATION
               scientificName                                                                                     Frangula dodonei subsp. dodonei
                    taxonRank                                                                                                          SUBSPECIES
                      kingdom                                                                                               

### 4. Technical Constraints: Search vs Registered Download
1. **Search API Deep-Pagination Wall**: The search API permits a maximum page size of `limit=300` and a maximum cumulative offset of `offset=100,000`. Querying beyond record 100,000 via search API returns an HTTP 400 error. Because our study landscapes contain over **4.7 million records**, full data acquisition must utilize the **GBIF Occurrence Download API**.
2. **Download DOI & Citation Requirements**: Every registered download request generates a persistent Digital Object Identifier (DOI) (e.g. `10.15468/dl.xxxxxx`), captures an immutable snapshot of all underlying datasets, and compiles complete attribution for all publishing institutions.

In [5]:
# Verification of download documentation parameters
download_formats = {
    "DWCA": "Darwin Core Archive (complete relational zip with occurrence.txt, verbatim.txt, multimedia.txt)",
    "SIMPLE_CSV": "Tab-delimited text extract containing core Darwin Core terms (~50% smaller)",
    "SPECIES_LIST": "Unique list of scientific taxa recorded in the bounding geometry"
}

print("Supported GBIF Download Formats:")
for fmt, desc in download_formats.items():
    print(f" - {fmt:12s}: {desc}")

Supported GBIF Download Formats:
 - DWCA        : Darwin Core Archive (complete relational zip with occurrence.txt, verbatim.txt, multimedia.txt)
 - SIMPLE_CSV  : Tab-delimited text extract containing core Darwin Core terms (~50% smaller)
 - SPECIES_LIST: Unique list of scientific taxa recorded in the bounding geometry
